In [35]:
import numpy as np 
import pandas as pd 

df = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/Divar.csv")

/tmp/ipykernel_79169/2050151770.py:4: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/Divar.csv")


In [36]:
cols = ['created_at_month', 'location_latitude', 'location_longitude' ]

In [37]:
#df[cols].head()

In [38]:
#print(df[cols].dtypes)

In [39]:
#print(df[cols].isnull().sum() / len(df[cols]) * 100)

In [40]:
#print(df[cols].nunique())

In [41]:
#for item in cols : 
#    print(df[item].value_counts())
#    print("---------------------")

In [42]:
#df[cols].describe()

In [43]:
#location_radius
#بهتر هست که این ستون رو پاک کنیم چون هم دسته های کمی داره و هم مقادیر خالی زیادی ( بالای 60 درصد ) رو شامل میشه 

df = df.drop(columns=["location_radius"])

In [44]:
# محدوده تقریبی جغرافیایی ایران
lat_min, lat_max = 25.0, 40.0
lon_min, lon_max = 44.0, 63.0



valid_lat = df['location_latitude'].between(lat_min, lat_max)
valid_lon = df['location_longitude'].between(lon_min, lon_max)

valid_coord = valid_lat & valid_lon


df.loc[~valid_coord, ['location_latitude', 'location_longitude']] = np.nan


print(df[['location_latitude', 'location_longitude']].isnull().sum())

location_latitude     344439
location_longitude    344439
dtype: int64


In [45]:
#location_utm_y
#location_utm_x
#اضافه کردن دو ستون جدید به دیتافریم 
import utm

def convert_to_utm(row):
    lat = row['location_latitude']
    lon = row['location_longitude']
    
    if pd.isnull(lat) or pd.isnull(lon):
        return pd.Series([np.nan, np.nan])
    
    try:
        utm_x, utm_y, _, _ = utm.from_latlon(lat, lon, force_zone_number=39)
        return pd.Series([utm_x, utm_y])
    except Exception:
        return pd.Series([np.nan, np.nan])

df[['location_utm_x', 'location_utm_y']] = df.apply(convert_to_utm, axis=1)

In [46]:
#test 
print(df[['location_utm_x', 'location_utm_y']].isnull().sum())

print(df[['location_latitude', 'location_longitude', 'location_utm_x', 'location_utm_y']].dropna().head())

location_utm_x    344439
location_utm_y    344439
dtype: int64
    location_latitude  location_longitude  location_utm_x  location_utm_y
0           35.811684           50.936600   494272.329998    3.963064e+06
2           35.703865           51.373459   533784.424602    3.951168e+06
7           35.729832           51.505466   545711.558185    3.954101e+06
8           35.712364           50.794781   481437.130969    3.952066e+06
10          35.778664           51.757549   568467.028494    3.959664e+06


In [47]:
# change type of column to datetime
df['created_at_month'] = pd.to_datetime(df['created_at_month'], errors='coerce')

In [48]:
# extract Year and Month 
# creat two column ["created_year", "created_month"]

df['created_year'] = df['created_at_month'].dt.year
df['created_month'] = df['created_at_month'].dt.month

In [49]:
#check all month and year is not null 
print(df[['created_year', 'created_month']].isnull().sum())

created_year     0
created_month    0
dtype: int64


In [ ]:
#delete this column ["created_at_month"]
df = df.drop(columns=['created_at_month'])

In [51]:
"""
=============================================================================
خلاصه تغییرات انجام‌شده روی ویژگی‌های زمان و مکان (time & location)
=============================================================================

۱. ستون‌های حذف‌شده (Dropped Columns):
-----------------------------------------------------------------------------
- location_radius   : به دلیل ۶۶٪ داده خالی (Null) و عدم کاربرد در مدل‌سازی حذف شد.
- created_at_month  : به دلیل فرمت متنی (String) حذف و به دو ویژگی عددی تبدیل شد.

۲. ستون‌های ویرایش‌شده (Cleaned / Masked Columns):
-----------------------------------------------------------------------------
- location_latitude : داده‌های خارج از محدوده ایران (خارج از ۲۵ تا ۴۰) به NaN تبدیل شدند.
- location_longitude: داده‌های خارج از محدوده ایران (خارج از ۴۴ تا ۶۳) به NaN تبدیل شدند.

۳. ستون‌های جدید ساخته‌شده (Feature Engineering):
-----------------------------------------------------------------------------
- location_utm_x    : مختصات X متری (بر مبنای UTM Zone 39 یکپارچه برای کل ایران).
- location_utm_y    : مختصات Y متری (بر مبنای UTM Zone 39 یکپارچه برای کل ایران).
- created_year      : سال ثبت آگهی به صورت عددی (مثلاً 2024 یا 2025).
- created_month     : شماره ماه ثبت آگهی به صورت عددی (بین ۱ تا ۱۲).
=============================================================================
"""

'\n=============================================================================\nخلاصه تغییرات انجام\u200cشده روی ویژگی\u200cهای زمان و مکان (time & location)\n=============================================================================\n\n۱. ستون\u200cهای حذف\u200cشده (Dropped Columns):\n-----------------------------------------------------------------------------\n- location_radius   : به دلیل ۶۶٪ داده خالی (Null) و عدم کاربرد در مدل\u200cسازی حذف شد.\n- created_at_month  : به دلیل فرمت متنی (String) حذف و به دو ویژگی عددی تبدیل شد.\n\n۲. ستون\u200cهای ویرایش\u200cشده (Cleaned / Masked Columns):\n-----------------------------------------------------------------------------\n- location_latitude : داده\u200cهای خارج از محدوده ایران (خارج از ۲۵ تا ۴۰) به NaN تبدیل شدند.\n- location_longitude: داده\u200cهای خارج از محدوده ایران (خارج از ۴۴ تا ۶۳) به NaN تبدیل شدند.\n\n۳. ستون\u200cهای جدید ساخته\u200cشده (Feature Engineering):\n----------------------------------------------------------